In [1]:
import pandas as pd
from pathlib import Path
import torch
import transformer_lens
import gc
import itertools
import random
from sklearn.svm import LinearSVC
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

#MODEL_NAME = "EleutherAI/pythia-12b"
MODEL_NAME = "microsoft/phi-2"


/home/pweiss/Research/mats/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading the model

if "model" in locals() or "model" in globals():
    del model

torch.cuda.empty_cache()

gc.collect()

# 4. Verify memory is cleared
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

# Load the Pythia-2.8b model
# This will download the model weights if they are not already cached.
model = transformer_lens.HookedTransformer.from_pretrained_no_processing(
    MODEL_NAME,
    # device="cpu",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

print(f"{MODEL_NAME} model loaded successfully.")

GPU memory allocated: 0.00 MB
GPU memory reserved: 0.00 MB


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.54it/s]


Loaded pretrained model microsoft/phi-2 into HookedTransformer
microsoft/phi-2 model loaded successfully.


In [ ]:
prompt = f"Q: In a single word, in what phase is TNT at room temperature?\nA:"
# prompt = "What is a more common name for Dihydrogenmonooxide?\nA:"
tokens = model.to_tokens(prompt)
print(f"Prompt tokens ({len(tokens[0])}): {tokens}\n")

response = model.generate(prompt, max_new_tokens=10, temperature=0.0)
print(f"Model response: {response}")

# Get the logits for the next token
logits = model(prompt)
print(f"Logits shape: {logits.shape}")
logits = logits[:, -1, :]  # Focus on the last token's logits

# Get the top 10 most likely tokens
top_k = 10
top_logits, top_indices = torch.topk(logits, top_k)

# Convert token indices to actual tokens
top_tokens = [model.to_string(idx) for idx in top_indices[0]]

# Display the results
print(f"Top {top_k} most likely next tokens:")
for i, (token, logit) in enumerate(zip(top_tokens, top_logits[0])):
    print(f"{i+1}. '{token}' (logit: {logit:.4f})")

Prompt tokens (23): tensor([[50256,    48,    25,   554,   257,  2060,  1573,    11,   287,   644,
          7108,   318,   555, 36909,   310,  1505,   379,  2119,  5951,    30,
           198,    32,    25]], device='cuda:0')



100%|██████████| 10/10 [00:00<00:00, 38.66it/s]

Model response: Q: In a single word, in what phase is ununoctium at room temperature?
A: Solid
Input: 
A: Ununo
Logits shape: torch.Size([1, 23, 51200])
Top 10 most likely next tokens:
1. ' Solid' (logit: 16.8750)
2. ' Liquid' (logit: 15.4375)
3. ' Gas' (logit: 14.1875)
4. ' solid' (logit: 14.0625)
5. ' Un' (logit: 13.7500)
6. ' Plasma' (logit: 13.1875)
7. ' Is' (logit: 13.1250)
8. ' The' (logit: 13.1250)
9. ' Unknown' (logit: 12.8750)
10. ' Ion' (logit: 12.6250)


In [ ]:
chemical_compounds = []
with open('wikidata_chemical_compounds.csv', 'r') as file:
    for line in file:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            chemical_compounds.append(parts[1])
chemical_compounds = chemical_compounds[1:] # Skip header
print(f"Total chemical compounds: {len(chemical_compounds)}")
print(f"First 10 compounds: {chemical_compounds[:10]}")
random.shuffle(chemical_compounds)
answers = {}
for compound in chemical_compounds[:1000]:
    prompt = f"Q: In a single word, in what phase state is {compound} at room temperature?\nA:"
    response = model.generate(prompt, max_new_tokens=1, temperature=0.0)
    completion = response.split("A:")[-1].strip()  # Get the first word after "A:"
    answers[completion] = answers.get(completion, 0) + 1
    print(f"{compound}: {response}")


Total chemical compounds: 147275
First 10 compounds: ['Streptokinase', 'Scarlet GN', 'Rubidium hexafluorotitanate', 'Claziprotamide', 'Peregal O', 'Vicasinabin', 'PR-000608', 'Fluspidine', 'Aluminosilicate Refractory Ceramic Fibres', 'Lysergic acid pyrrolinide']


100%|██████████| 1/1 [00:00<00:00, 23.69it/s]


epelsiban besylate: Q: In a single word, in what phase state is epelsiban besylate at room temperature?
A: In


100%|██████████| 1/1 [00:00<00:00, 25.59it/s]


xanthiol hydrochloride: Q: In a single word, in what phase state is xanthiol hydrochloride at room temperature?
A: Solid


...

100%|██████████| 1/1 [00:00<00:00, 29.55it/s]


"1: Q: In a single word, in what phase state is "1 at room temperature?
A: Solid


100%|██████████| 1/1 [00:00<00:00, 29.45it/s]


"N: Q: In a single word, in what phase state is "N at room temperature?
A: Solid


KeyboardInterrupt: 

In [ ]:
print("\nAnswer distribution:")
for phase, count in answers.items():
    print(f"{phase}: {count}")


Answer distribution:
Solid: 665
Liquid: 309
Diaz: 2
Tin: 1
The: 6
Gas: 6
Pent: 1
Ox: 1
Nit: 1
Bent: 1
D: 1
Ang: 1
Flu: 3
In: 1
S: 1
